In [1]:
import os
os.system('pip install -q pyvi')

import re
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
    AutoConfig
)
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from pyvi import ViTokenizer
from tqdm import tqdm
import torch.nn.functional as F


# 1) CONFIG
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

class Config:
    model_name = 'vinai/phobert-base'
    max_len = 200  # Tăng lên 200
    batch_size = 32  # Tăng batch size
    epochs = 8  # Tăng số epochs
    lr = 2e-5
    weight_decay = 0.01
    n_folds = 5
    warmup_ratio = 0.1
    gradient_accumulation_steps = 2  # Thêm gradient accumulation
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = Config()
print('Device:', cfg.device)


# 2) LOAD DATA
train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')

TEXT_COL = 'text' if 'text' in train_df.columns else train_df.columns[1]
LABEL_COL = 'sentiment'


# 3) PREPROCESS - CẢI THIỆN
def clean_text(text):
    text = str(text).lower().strip()
    # Loại bỏ URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Loại bỏ mentions và hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Loại bỏ ký tự đặc biệt nhưng giữ dấu tiếng Việt
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Xử lý từ viết tắt
    text = re.sub(r'\bko\b', 'không', text)
    text = re.sub(r'\bok\b', 'ổn', text)
    text = re.sub(r'\bkg\b', 'không', text)
    text = re.sub(r'\bdc\b', 'được', text)
    text = re.sub(r'\bvs\b', 'với', text)
    text = re.sub(r'\bwa\b', 'quá', text)
    return ViTokenizer.tokenize(text)

train_df[TEXT_COL] = train_df[TEXT_COL].fillna('').apply(clean_text)
test_df[TEXT_COL] = test_df[TEXT_COL].fillna('').apply(clean_text)


# 4) DATA AUGMENTATION
def augment_text(text):
    """Simple text augmentation by adding noise"""
    words = text.split()
    if len(words) < 5:
        return text
    
    # Randomly drop words (10% chance)
    if random.random() < 0.1:
        words = [w for w in words if random.random() > 0.1]
    
    # Randomly duplicate some words
    if random.random() < 0.1:
        new_words = []
        for w in words:
            new_words.append(w)
            if random.random() < 0.05:
                new_words.append(w)
        words = new_words
    
    return ' '.join(words)


# 5) TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels=None, augment=False):
        self.texts = texts
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        if self.augment and self.labels is not None:
            text = augment_text(text)
        
        enc = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=cfg.max_len,
            return_tensors='pt'
        )
        item = {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0)
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


# 6) IMPROVED MODEL with additional layers
class ImprovedPhoBERTClassifier(nn.Module):
    def __init__(self, model_name, n_classes, dropout_rate=0.3):
        super().__init__()
        config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name, config=config)
        hidden = self.bert.config.hidden_size
        
        # Thêm các layers phức tạp hơn
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        # Multi-head attention pooling
        self.attention = nn.MultiheadAttention(hidden, num_heads=8, batch_first=True)
        
        # Additional dense layers
        self.dense1 = nn.Linear(hidden, hidden // 2)
        self.dense2 = nn.Linear(hidden // 2, hidden // 4)
        self.fc = nn.Linear(hidden // 4, n_classes)
        
        self.layer_norm = nn.LayerNorm(hidden)
        self.activation = nn.GELU()
        
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Get all hidden states
        last_hidden = out.last_hidden_state  # [batch, seq_len, hidden]
        
        # Apply multi-head attention
        attn_out, _ = self.attention(last_hidden, last_hidden, last_hidden, 
                                      key_padding_mask=(attention_mask == 0))
        
        # Add residual connection
        attn_out = attn_out + last_hidden
        attn_out = self.layer_norm(attn_out)
        
        # Use attention output
        cls = attn_out[:, 0, :]
        
        x = self.dropout1(cls)
        x = self.dense1(x)
        x = self.activation(x)
        x = self.dropout2(x)
        x = self.dense2(x)
        x = self.activation(x)
        x = self.dropout2(x)
        
        return self.fc(x)


# 7) IMPROVED FOCAL LOSS
class ImprovedFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.5, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# 8) TRAIN FUNCTION with early stopping
def train_one_fold(train_idx, val_idx, fold):
    x_train = train_df.iloc[train_idx][TEXT_COL].tolist()
    y_train = train_df.iloc[train_idx][LABEL_COL].tolist()
    x_val = train_df.iloc[val_idx][TEXT_COL].tolist()
    y_val = train_df.iloc[val_idx][LABEL_COL].tolist()

    train_ds = SentimentDataset(x_train, y_train, augment=True)
    val_ds = SentimentDataset(x_val, y_val, augment=False)
    test_ds = SentimentDataset(test_df[TEXT_COL].tolist())

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size)

    model = ImprovedPhoBERTClassifier(cfg.model_name, 3, dropout_rate=0.35).to(cfg.device)

    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(cfg.device)

    criterion = ImprovedFocalLoss(alpha=class_weights, gamma=2.5)
    
    # Group parameters for differential learning rates
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 
         'weight_decay': cfg.weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 
         'weight_decay': 0.0}
    ]
    
    optimizer = AdamW(optimizer_grouped_parameters, lr=cfg.lr)

    total_steps = len(train_loader) * cfg.epochs // cfg.gradient_accumulation_steps
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * cfg.warmup_ratio),
        num_training_steps=total_steps
    )

    best_f1 = 0
    best_test_probs = None
    patience_counter = 0
    max_patience = 2  # Early stopping patience

    for epoch in range(cfg.epochs):
        model.train()
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc=f'Fold {fold} Epoch {epoch+1}')
        optimizer.zero_grad()
        
        for i, batch in enumerate(progress_bar):
            ids = batch['input_ids'].to(cfg.device)
            mask = batch['attention_mask'].to(cfg.device)
            labels = batch['labels'].to(cfg.device)

            logits = model(ids, mask)
            loss = criterion(logits, labels)
            loss = loss / cfg.gradient_accumulation_steps
            loss.backward()
            total_loss += loss.item() * cfg.gradient_accumulation_steps
            
            if (i + 1) % cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            
            progress_bar.set_postfix({'loss': total_loss / (i + 1)})
        
        # Validation
        model.eval()
        preds, truths = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids = batch['input_ids'].to(cfg.device)
                mask = batch['attention_mask'].to(cfg.device)
                labels = batch['labels'].to(cfg.device)

                logits = model(ids, mask)
                pred = torch.argmax(logits, dim=1)
                preds.extend(pred.cpu().numpy())
                truths.extend(labels.cpu().numpy())

        f1 = f1_score(truths, preds, average='macro')
        print(f'Fold {fold} Epoch {epoch+1} Macro F1: {f1:.5f}')

        # Early stopping and model saving
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            probs = []
            with torch.no_grad():
                for batch in test_loader:
                    ids = batch['input_ids'].to(cfg.device)
                    mask = batch['attention_mask'].to(cfg.device)
                    logits = model(ids, mask)
                    prob = torch.softmax(logits, dim=1)
                    probs.append(prob.cpu().numpy())
            best_test_probs = np.vstack(probs)
        else:
            patience_counter += 1
            if patience_counter >= max_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return best_f1, best_test_probs


# 9) K-FOLD TRAINING
skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=42)
all_test_probs = []
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df[LABEL_COL]), 1):
    print(f"\n{'='*50}")
    print(f"Training Fold {fold}")
    print(f"{'='*50}")
    score, test_probs = train_one_fold(train_idx, val_idx, fold)
    fold_scores.append(score)
    all_test_probs.append(test_probs)

print(f'\nCV Macro F1: {np.mean(fold_scores):.6f} (+/- {np.std(fold_scores):.6f})')


# 10) ENSEMBLE WITH WEIGHTS
# Weight folds by their performance
weights = np.array(fold_scores) / np.sum(fold_scores)
weighted_probs = np.zeros_like(all_test_probs[0])
for i, probs in enumerate(all_test_probs):
    weighted_probs += probs * weights[i]

preds = np.argmax(weighted_probs, axis=1)

submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': preds
})
submission.to_csv('submission.csv', index=False)
print('submission.csv saved!')
print(submission.head())

# Print detailed results
print(f"\nDetailed Results:")
print(f"Fold scores: {fold_scores}")
print(f"Mean: {np.mean(fold_scores):.6f}")
print(f"Std: {np.std(fold_scores):.6f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.8 MB/s eta 0:00:00
Device: cuda


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fold 5 Epoch 6 Macro F1: 0.82081


Fold 5 Epoch 7: 100%|██████████| 284/284 [05:31<00:00,  1.17s/it, loss=0.0635]


Fold 5 Epoch 7 Macro F1: 0.82493


Fold 5 Epoch 8: 100%|██████████| 284/284 [05:31<00:00,  1.17s/it, loss=0.0508]


Fold 5 Epoch 8 Macro F1: 0.82765

CV Macro F1: 0.829659 (+/- 0.005917)
submission.csv saved!
   id  sentiment
0   0          0
1   1          1
2   2          2
3   3          2
4   4          0

Detailed Results:
Fold scores: [0.8287910967122508, 0.8235904793078362, 0.8409657952412951, 0.8272954315018225, 0.8276529118098072]
Mean: 0.829659
Std: 0.005917
